In [1]:
# Neural Identifier Training - 2-DOF Planar Manipulator
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (2-DOF Robot Arm)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a 2-link planar robot arm.
    state = [q1, q2, dq1, dq2] (Angles and Velocities)
    u     = [tau1, tau2]       (Torques)
    """
    # Robot Parameters
    m1, m2 = 1.0, 1.0  # Mass (kg)
    l1, l2 = 1.0, 1.0  # Lengths (m)
    g = 9.81
    
    q1, q2, dq1, dq2 = state
    tau1, tau2 = u

    # --- Mass Matrix M(q) ---
    c2 = np.cos(q2)
    s2 = np.sin(q2)
    
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * c2
    M12 = m2 * l2**2 + m2 * l1 * l2 * c2
    M21 = M12
    M22 = m2 * l2**2
    M = np.array([[M11, M12], [M21, M22]])

    # --- Coriolis/Centrifugal Matrix C(q, dq) ---
    h = -m2 * l1 * l2 * s2
    C11 = h * dq2
    C12 = h * (dq1 + dq2)
    C21 = -h * dq1
    C22 = 0.0
    C = np.array([[C11, C12], [C21, C22]])

    # --- Gravity Vector G(q) ---
    s1 = np.sin(q1)
    s12 = np.sin(q1 + q2)
    G1 = (m1 + m2) * g * l1 * s1 + m2 * g * l2 * s12
    G2 = m2 * g * l2 * s12
    G = np.array([G1, G2])

    # --- Equation of Motion: M*ddq + C*dq + G = tau ---
    damping = 0.5 * np.array([dq1, dq2])
    torque_vector = np.array([tau1, tau2])
    
    rhs = torque_vector - (C @ np.array([dq1, dq2])) - G - damping
    
    # Solve for accelerations
    ddq = np.linalg.solve(M, rhs)
    
    return np.concatenate(([dq1, dq2], ddq))

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-4):
    """
    Euler integration step.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Add small process noise
    noise = np.random.laplace(4) * process_noise_std
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=0.5):
    """Sigmoid S(z). Beta reduced to widen the active range."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for 2-DOF Arm - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [q1, q2, dq1, dq2]
    u_input = [tau1, tau2]
    neuron_index: índice de la neurona (0=q1, 1=q2, 2=dq1, 3=dq2)
    
    Cada neurona puede tener su propia estructura de características.
    """
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    # Términos básicos sigmoidales
    s_q1 = sigmoidal(q1)
    s_q2 = sigmoidal(q2)
    s_dq1 = sigmoidal(dq1)
    s_dq2 = sigmoidal(dq2)
    
    # Términos trigonométricos (útiles para dinámica de robots)
    sin_q1 = np.sin(q1)
    sin_q2 = np.sin(q2)
    cos_q1 = np.cos(q1)
    cos_q2 = np.cos(q2)
    sin_q1q2 = np.sin(q1 + q2)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para q1 (ángulo articulación 1)
        return np.array([
            s_q1,                      # Estado actual
            s_dq1,                     # Velocidad actual
            s_q1 * s_dq1,             # Interacción ángulo-velocidad
            s_q1**2,                   # Término cuadrático
            s_q2,                      # Acoplamiento con q2
            # sin_q1,                    # Término gravitacional
            tau1 * 0.1,               # Entrada de control
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para q2 (ángulo articulación 2)
        return np.array([
            s_q2,                      # Estado actual
            s_dq2,                     # Velocidad actual
            s_q2 * s_dq2,             # Interacción ángulo-velocidad
            s_q2**3,                   # Término cúbico (mayor no linealidad)
            s_q1 * s_q2,              # Acoplamiento con q1
            # sin_q2,                    # Término gravitacional
            # sin_q1q2,                  # Acoplamiento cinemático
            tau2 * 0.1,               # Entrada de control
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 2:  # Neurona para dq1 (velocidad articulación 1)
        return np.array([
            s_dq1,                     # Estado actual
            s_dq1**2,                  # Término cuadrático (fricción)
            s_q1,                      # Dependencia del ángulo
            s_q2 * s_dq1,             # Coriolis proxy
            s_q2 * s_dq2,             # Coriolis proxy
            s_dq2,                     # Acoplamiento velocidades
            # cos_q2,                    # Términos de masa variable
            tau1 * 0.1,               # Entrada de control (lineal)
            tau2 * 0.05,              # Acoplamiento de torques
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 3:  # Neurona for dq2 (velocidad articulación 2)
        return np.array([
            s_dq2,                     # Estado actual
            s_dq2**2,                  # Término cuadrático (fricción)
            s_q2,                      # Dependencia del ángulo
            s_q1 * s_dq2,             # Coriolis proxy
            s_dq1 * s_dq2,            # Interacción velocidades
            s_dq1,                     # Acoplamiento con dq1
            # cos_q2,                    # Términos de masa variable
            tau2 * 0.1,               # Entrada de control (lineal)
            tau1 * 0.05,              # Acoplamiento de torques
            # 1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:
        return 6
    elif neuron_index == 1:
        return 6
    elif neuron_index == 2:
        return 8
    elif neuron_index == 3:
        return 8
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-4, R=1e-2):
        self.n_neurons = n_neurons
        # Cada neurona tiene su propio número de pesos
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            H = z.reshape(-1, 1)
            
            # Predict P
            P_pred = self.P[i] + self.Q_matrices[i]
            
            # Kalman Gain
            S = self.R + (H.T @ P_pred @ H)[0,0]
            K = (P_pred @ H).flatten() / S
            
            # Error
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            
            # Update Weights
            self.weights[i] += self.eta * K * err
            
            # Update Covariance (Joseph form)
            I_KH = np.eye(len(z)) - np.outer(K, z)
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K, K)*self.R

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*1e-4 for i in range(n_neurons)]
        self.R = 1e-2
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params (calculados para cada neurona)
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_particles=800):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        # Cada neurona tiene partículas de diferente dimensión
        self.particles = [np.random.randn(n_particles, get_z_size(i))*0.2 
                         for i in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.Q_std = 0.2
        self.R_std = 0.1

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n_weights = get_z_size(i)
            
            # 1. Drift
            self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.Q_std
            
            # 2. Weight
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            likelihood = np.exp(-0.5 * (err/self.R_std)**2)
            self.weights_pf[i] *= (likelihood + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/2:
                indices = np.random.choice(self.n_particles, self.n_particles, 
                                         p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) 
                for i in range(self.n_neurons)]

# ============================================================
# 4) Simulation Main Loop
# ============================================================
if __name__ == "__main__":
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4
    
    # Init Trainers
    ekf = EKF_Trainer(n_states, eta=1.0)
    ukf = UKF_Trainer(n_states, eta=0.9)
    pf = PF_Trainer(n_states, n_particles=800)
    
    # Arrays
    x_true = np.zeros((n_steps, 4))
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Initial Conditions
    x_true[0] = [-np.pi/2, 0, 0, 0] 
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0]  = x_true[0]
    
    # Excitation Input
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        tau1 = 30.0 * np.sin(2.0 * t[k]) 
        tau2 = 15.0 * np.cos(3.0 * t[k])
        u_hist[k] = [tau1, tau2]

    print("Simulating 2-DOF Manipulator with neuron-specific features...")
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i}: {get_z_size(i)} características")
    
    for k in range(n_steps - 1):
        # 1. Physics Step
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Identify
        # EKF
        ekf.update(x_true[k+1], x_true[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_true[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            
        # UKF
        ukf.update(x_true[k+1], x_true[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_true[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            
        # PF
        pf.update(x_true[k+1], x_true[k], u_hist[k])
        w_pf = pf.get_estimates()
        for i in range(4): 
            z = construct_z_vector(x_true[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z)
            
        if k % 100 == 0: print(f"Step {k}")

    # ============================================================
    # 5) Visualización - Formato Tesis
    # ============================================================
    
    print("\nGenerando visualizaciones...")
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # --- Calcular MSE ---
    mse_q1_ekf = np.mean((x_true[:, 0] - x_est_ekf[:, 0])**2)
    mse_q2_ekf = np.mean((x_true[:, 1] - x_est_ekf[:, 1])**2)
    mse_dq1_ekf = np.mean((x_true[:, 2] - x_est_ekf[:, 2])**2)
    mse_dq2_ekf = np.mean((x_true[:, 3] - x_est_ekf[:, 3])**2)
    
    mse_q1_ukf = np.mean((x_true[:, 0] - x_est_ukf[:, 0])**2)
    mse_q2_ukf = np.mean((x_true[:, 1] - x_est_ukf[:, 1])**2)
    mse_dq1_ukf = np.mean((x_true[:, 2] - x_est_ukf[:, 2])**2)
    mse_dq2_ukf = np.mean((x_true[:, 3] - x_est_ukf[:, 3])**2)
    
    mse_q1_pf = np.mean((x_true[:, 0] - x_est_pf[:, 0])**2)
    mse_q2_pf = np.mean((x_true[:, 1] - x_est_pf[:, 1])**2)
    mse_dq1_pf = np.mean((x_true[:, 2] - x_est_pf[:, 2])**2)
    mse_dq2_pf = np.mean((x_true[:, 3] - x_est_pf[:, 3])**2)
    
    mse_total_ekf = mse_q1_ekf + mse_q2_ekf + mse_dq1_ekf + mse_dq2_ekf
    mse_total_ukf = mse_q1_ukf + mse_q2_ukf + mse_dq1_ukf + mse_dq2_ukf
    mse_total_pf = mse_q1_pf + mse_q2_pf + mse_dq1_pf + mse_dq2_pf
    
    # Reporte
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)
    
    print("\n--- Comparación de Desempeño (MSE) - Manipulador 2-DOF ---")
    print(f"EKF MSE q₁:   {mse_q1_ekf:.6f} | MSE dq₁:   {mse_dq1_ekf:.6f}")
    print(f"EKF MSE q₂:   {mse_q2_ekf:.6f} | MSE dq₂:   {mse_dq2_ekf:.6f}")
    print(f"UKF MSE q₁:   {mse_q1_ukf:.6f} | MSE dq₁:   {mse_dq1_ukf:.6f}")
    print(f"UKF MSE q₂:   {mse_q2_ukf:.6f} | MSE dq₂:   {mse_dq2_ukf:.6f}")
    print(f"PF  MSE q₁:   {mse_q1_pf:.6f} | MSE dq₁:   {mse_dq1_pf:.6f}")
    print(f"PF  MSE q₂:   {mse_q2_pf:.6f} | MSE dq₂:   {mse_dq2_pf:.6f}")
    
    # --- Gráficas por Estado ---
    states_info = [
        {'idx': 0, 'var': 'q₁', 'desc': 'Ángulo Articulación 1', 'y_label': 'Ángulo q₁ (rad)'},
        {'idx': 1, 'var': 'q₂', 'desc': 'Ángulo Articulación 2', 'y_label': 'Ángulo q₂ (rad)'},
        {'idx': 2, 'var': 'dq₁', 'desc': 'Velocidad Articulación 1', 'y_label': 'Velocidad dq₁ (rad/s)'},
        {'idx': 3, 'var': 'dq₂', 'desc': 'Velocidad Articulación 2', 'y_label': 'Velocidad dq₂ (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=t, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Manipulador 2-DOF',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # --- Gráfica de errores combinada ---
    fig_err = go.Figure()
    
    errors_info = [
        {'idx': 0, 'label': 'q₁', 'mse_ekf': mse_q1_ekf, 'mse_ukf': mse_q1_ukf, 'mse_pf': mse_q1_pf},
        {'idx': 1, 'label': 'q₂', 'mse_ekf': mse_q2_ekf, 'mse_ukf': mse_q2_ukf, 'mse_pf': mse_q2_pf}
    ]
    
    for err_info in errors_info:
        i = err_info['idx']
        label = err_info['label']
        
        error_ekf = x_true[:, i] - x_est_ekf[:, i]
        error_ukf = x_true[:, i] - x_est_ukf[:, i]
        error_pf = x_true[:, i] - x_est_pf[:, i]
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_ekf,
            mode='lines',
            name=f'EKF Error {label} (MSE={err_info["mse_ekf"]:.2e})',
            line=dict(color='#1f77b4', width=1.5),
            opacity=0.8
        ))
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_ukf,
            mode='lines',
            name=f'UKF Error {label} (MSE={err_info["mse_ukf"]:.2e})',
            line=dict(color='#2ca02c', width=1.5, dash='dot'),
            opacity=0.8
        ))
        
        fig_err.add_trace(go.Scatter(
            x=t, y=error_pf,
            mode='lines',
            name=f'PF Error {label} (MSE={err_info["mse_pf"]:.2e})',
            line=dict(color='#d62728', width=1.5, dash='dashdot'),
            opacity=0.8
        ))
    
    fig_err.update_layout(
        title={
            'text': 'Errores de Estimación de Ángulos - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='Error de Estimación (rad)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_err.show()
    
    # --- Gráfica de barras comparando MSE ---
    fig_mse = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_mse.add_trace(go.Bar(
        name='Ángulo q₁',
        x=filters,
        y=[mse_q1_ekf, mse_q1_ukf, mse_q1_pf],
        marker_color='#636EFA',
        text=[f'{mse_q1_ekf:.2e}', f'{mse_q1_ukf:.2e}', f'{mse_q1_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Ángulo q₂',
        x=filters,
        y=[mse_q2_ekf, mse_q2_ukf, mse_q2_pf],
        marker_color='#EF553B',
        text=[f'{mse_q2_ekf:.2e}', f'{mse_q2_ukf:.2e}', f'{mse_q2_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Velocidad dq₁',
        x=filters,
        y=[mse_dq1_ekf, mse_dq1_ukf, mse_dq1_pf],
        marker_color='#00CC96',
        text=[f'{mse_dq1_ekf:.2e}', f'{mse_dq1_ukf:.2e}', f'{mse_dq1_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Velocidad dq₂',
        x=filters,
        y=[mse_dq2_ekf, mse_dq2_ukf, mse_dq2_pf],
        marker_color='#AB63FA',
        text=[f'{mse_dq2_ekf:.2e}', f'{mse_dq2_ukf:.2e}', f'{mse_dq2_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE) - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_mse.show()
    
    print("\n✅ Visualización completa.")

Simulating 2-DOF Manipulator with neuron-specific features...

Estructura de características por neurona:
  Neurona 0: 6 características
  Neurona 1: 6 características
  Neurona 2: 8 características
  Neurona 3: 8 características
Step 0
Step 100
Step 200
Step 300
Step 400
Step 500
Step 600
Step 700
Step 800
Step 900

Generando visualizaciones...

🏆 MEJOR FILTRO: PF (MSE total: 0.516247)

--- Comparación de Desempeño (MSE) - Manipulador 2-DOF ---
EKF MSE q₁:   0.019814 | MSE dq₁:   0.260189
EKF MSE q₂:   0.359601 | MSE dq₂:   1.692738
UKF MSE q₁:   1.229587 | MSE dq₁:   1.770500
UKF MSE q₂:   4.648888 | MSE dq₂:   8.923501
PF  MSE q₁:   0.001262 | MSE dq₁:   0.002243
PF  MSE q₂:   0.006068 | MSE dq₂:   0.506675



✅ Visualización completa.
